# Camelot Wheel — Harmonic Journey (paper figure)

DJ set 진행을 Camelot Wheel 위에 정방형, 논문 차트 스타일로 시각화.

- **입력**: rekordbox에서 export한 playlist `.txt` (tab-separated, 보통 UTF-16) 또는 `Track Title / Artist / Key` 컬럼의 DataFrame
- **출력**: 타임스탬프 폴더 안에 `.png` (300dpi, 정방형), `.pdf` (벡터), `.csv`

**Wheel 좌표**: 12 시 방향 = 12, 시계 방향으로 1→11.  
**Ring**: 안쪽 = `A` (minor), 바깥쪽 = `B` (major).

**라벨**: 각 점 옆에 `n. Artist — Track Title` 형태로 표시, `adjustText`로 자동 위치 조정.

## 1. Imports & rcParams

In [ ]:
import os
import re
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import patheffects
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from adjustText import adjust_text   # pip install adjustText

# Paper-friendly defaults + vector output that stays editable in Illustrator
mpl.rcParams.update({
    # Coding/editor-style monospace font (Menlo = VSCode/Xcode default on macOS),
    # with Korean + CJK fallbacks so non-Latin track titles render instead of
    # tofu boxes. Pass the list directly to font.family (NOT the "monospace"
    # generic) — that is what enables matplotlib's per-glyph font fallback.
    "font.family":       ["Menlo", "Monaco", "DejaVu Sans Mono",
                          "Apple SD Gothic Neo",   # Korean
                          "Hiragino Sans GB",      # Chinese / extra CJK
                          "Consolas", "Courier New"],
    "mathtext.fontset":  "stix",
    "axes.linewidth":    0.6,
    "axes.edgecolor":    "#222222",
    "text.color":        "#1a1a1a",
    "savefig.facecolor": "white",
    # --- keep text as text (not outlined paths) in vector files ---
    "svg.fonttype":      "none",   # <text> elements in SVG
    "pdf.fonttype":      42,       # TrueType embed in PDF
    "ps.fonttype":       42,
    "pdf.use14corefonts": False,
})

# wheel geometry
R_INNER = 0.62   # A (minor)
R_OUTER = 1.00   # B (major)

In [ ]:
recordbox_history = os.path("")

## 2. Key → Camelot

In [ ]:
KEY_TO_CAMELOT = {
    "Abm": "1A",  "G#m": "1A",
    "Ebm": "2A",  "D#m": "2A",
    "Bbm": "3A",  "A#m": "3A",
    "Fm":  "4A",
    "Cm":  "5A",
    "Gm":  "6A",
    "Dm":  "7A",
    "Am":  "8A",
    "Em":  "9A",
    "Bm":  "10A",
    "F#m": "11A", "Gbm": "11A",
    "C#m": "12A", "Dbm": "12A",
    "B":   "1B",
    "F#":  "2B",  "Gb":  "2B",
    "Db":  "3B",  "C#":  "3B",
    "Ab":  "4B",  "G#":  "4B",
    "Eb":  "5B",  "D#":  "5B",
    "Bb":  "6B",  "A#":  "6B",
    "F":   "7B",
    "C":   "8B",
    "G":   "9B",
    "D":   "10B",
    "A":   "11B",
    "E":   "12B",
}

def normalize_key(key_str):
    if key_str is None or (isinstance(key_str, float) and np.isnan(key_str)):
        return None
    s = str(key_str).strip()
    if not s:
        return None
    m = re.fullmatch(r"\s*(\d{1,2})\s*([AaBb])\s*", s)
    if m:
        n = int(m.group(1))
        if 1 <= n <= 12:
            return f"{n}{m.group(2).upper()}"
    s2 = s.replace("minor", "m").replace("Minor", "m")
    s2 = s2.replace("major", "").replace("Major", "")
    s2 = s2.replace(" ", "")
    return KEY_TO_CAMELOT.get(s2)


assert normalize_key("Am")  == "8A"
assert normalize_key("8A")  == "8A"
assert normalize_key("C")   == "8B"
assert normalize_key("F#m") == "11A"
print("normalize_key OK")

## 3. rekordbox `.txt` 파서

In [ ]:
def parse_rekordbox_txt(filepath):
    last_err = None
    for enc in ("utf-16", "utf-16-le", "utf-8-sig", "utf-8", "cp949"):
        try:
            df = pd.read_csv(filepath, sep="\t", encoding=enc)
            break
        except (UnicodeDecodeError, UnicodeError) as e:
            last_err = e
            continue
    else:
        raise last_err

    df.columns = [c.strip().lstrip("#").strip() for c in df.columns]
    key_col    = next((c for c in df.columns if c.lower() in ("key", "tonality")), None)
    title_col  = next((c for c in df.columns if c.lower() in ("track title", "title", "name")), None)
    artist_col = next((c for c in df.columns if c.lower() == "artist"), None)
    df = df.rename(columns={key_col: "Key", title_col: "Track Title", artist_col: "Artist"})
    df["_camelot"] = df["Key"].apply(normalize_key)
    return df

In [ ]:
df = parse_rekordbox_txt(recordbox_history)

## 4. Wheel 좌표 & overlap jitter

In [ ]:
def camelot_xy(camelot):
    if camelot is None:
        return None
    n = int(camelot[:-1])
    ring = camelot[-1]
    r = R_INNER if ring == "A" else R_OUTER
    theta = np.deg2rad(90 - (n % 12) * 30)
    return r * np.cos(theta), r * np.sin(theta)


def jitter_overlaps(points, jitter_r=0.045):
    counts, indices = defaultdict(int), []
    for x, y in points:
        k = (round(x, 4), round(y, 4))
        indices.append(counts[k])
        counts[k] += 1
    out = []
    for i, (x, y) in enumerate(points):
        k = (round(x, 4), round(y, 4))
        total = counts[k]
        if total > 1:
            sub = 2 * np.pi * indices[i] / total
            out.append((x + jitter_r * np.cos(sub), y + jitter_r * np.sin(sub)))
        else:
            out.append((x, y))
    return out

## 5. Plot

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# ===== three selectable visual designs =====================================
# Pick one by name when calling plot_harmonic_journey(..., style="paper").
STYLES = {
    # A. Paper — light, academic, viridis play-order gradient (the original look)
    "paper": dict(
        fig_bg="white", ax_bg="white",
        cmap="viridis",
        ring="#cfd2d6", spoke="#dcdee1", pos_label="#777a80",
        point_size=46, point_edge="#1a1a1a", point_edge_w=0.7,
        text="#1a1a1a", text_stroke="white", text_stroke_w=2.0,
        leader="#5a5d63", cbar_text="#1a1a1a", cbar_outline="#1a1a1a",
        glow=False,
    ),
    # B. Neon Night — dark club aesthetic, glowing plasma path, light labels
    "neon": dict(
        fig_bg="#0e1116", ax_bg="#0e1116",
        cmap="plasma",
        ring="#2b313c", spoke="#20242c", pos_label="#8b93a3",
        point_size=52, point_edge="#0e1116", point_edge_w=0.8,
        text="#eef2f8", text_stroke="#0e1116", text_stroke_w=2.6,
        leader="#5d6675", cbar_text="#cdd3de", cbar_outline="#cdd3de",
        glow=True,
    ),
    # C. Minimal Mono — editorial, single warm accent, hairline wheel
    "mono": dict(
        fig_bg="white", ax_bg="white",
        cmap=LinearSegmentedColormap.from_list(
            "warm_mono", ["#f2c1a0", "#e0673f", "#7d2414"]),
        ring="#e6e6e6", spoke="#f0f0f0", pos_label="#c0c0c0",
        point_size=40, point_edge="#222222", point_edge_w=0.6,
        text="#222222", text_stroke="white", text_stroke_w=2.0,
        leader="#cfcfcf", cbar_text="#555555", cbar_outline="#555555",
        glow=False,
    ),
}


def plot_harmonic_journey(df, output_path,
                          title="Harmonic Journey on the Camelot Wheel",
                          subtitle=None,
                          style="paper",
                          label_format="{artist} — {title}",
                          max_label_chars=None):
    """Square, paper-figure-style Camelot journey plot.

    style: one of STYLES — "paper", "neon", or "mono". Selects the design.
    max_label_chars is kept for backward compatibility but is ignored:
    track titles are never truncated.
    """
    if style not in STYLES:
        raise ValueError(f"unknown style {style!r}; choose from {list(STYLES)}")
    S = STYLES[style]

    df = df.reset_index(drop=True)
    valid = df.dropna(subset=["_camelot"]).reset_index(drop=True)

    raw_xy = [camelot_xy(c) for c in valid["_camelot"]]
    xy = jitter_overlaps(raw_xy)
    xs = np.array([p[0] for p in xy])
    ys = np.array([p[1] for p in xy])
    n_pts = len(xy)

    cmap = S["cmap"] if not isinstance(S["cmap"], str) else plt.get_cmap(S["cmap"])
    norm = Normalize(vmin=0, vmax=max(n_pts - 1, 1))

    # ---- square canvas (saved without bbox_inches='tight' to preserve ratio)
    fig = plt.figure(figsize=(9, 9), facecolor=S["fig_bg"])
    ax = fig.add_axes([0.02, 0.10, 0.96, 0.82])
    ax.set_facecolor(S["ax_bg"])
    ax.set_aspect("equal")
    LIM = 2.15
    ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM)
    ax.axis("off")

    # ---- wheel rings ----
    th = np.linspace(0, 2 * np.pi, 400)
    for r in [R_INNER - 0.07, R_INNER + 0.07, R_OUTER - 0.07, R_OUTER + 0.07]:
        ax.plot(r * np.cos(th), r * np.sin(th),
                color=S["ring"], lw=0.6, zorder=0)
    for n in range(12):
        a = np.deg2rad(90 - n * 30 - 15)
        ax.plot(
            [(R_INNER - 0.07) * np.cos(a), (R_OUTER + 0.07) * np.cos(a)],
            [(R_INNER - 0.07) * np.sin(a), (R_OUTER + 0.07) * np.sin(a)],
            color=S["spoke"], lw=0.45, zorder=0,
        )

    # ---- Camelot position labels: A inside, B outside ----
    R_LBL_A = R_INNER - 0.135
    R_LBL_B = R_OUTER + 0.135
    for n in range(1, 13):
        a = np.deg2rad(90 - (n % 12) * 30)
        ax.text(R_LBL_A * np.cos(a), R_LBL_A * np.sin(a), f"{n}A",
                ha="center", va="center", color=S["pos_label"], fontsize=8.5)
        ax.text(R_LBL_B * np.cos(a), R_LBL_B * np.sin(a), f"{n}B",
                ha="center", va="center", color=S["pos_label"], fontsize=8.5)

    # ---- connecting path with play-order gradient ----
    for i in range(n_pts - 1):
        c = cmap(norm(i))
        if S["glow"]:   # soft underglow for the dark design
            ax.plot([xs[i], xs[i + 1]], [ys[i], ys[i + 1]],
                    color=c, lw=5.5, alpha=0.18, solid_capstyle="round",
                    zorder=1)
        ax.annotate(
            "", zorder=2,
            xy=(xs[i + 1], ys[i + 1]),
            xytext=(xs[i], ys[i]),
            arrowprops=dict(arrowstyle="-|>", color=c, lw=1.1,
                            alpha=0.95, mutation_scale=10,
                            shrinkA=4.5, shrinkB=4.5),
        )

    # ---- points ----
    sc_colors = [cmap(norm(i)) for i in range(n_pts)]
    if S["glow"]:   # faint halo behind each point
        ax.scatter(xs, ys, s=S["point_size"] * 4.5, c=sc_colors,
                   alpha=0.16, linewidths=0, zorder=2)
    ax.scatter(xs, ys, s=S["point_size"], c=sc_colors,
               edgecolors=S["point_edge"], linewidths=S["point_edge_w"], zorder=3)

    # ---- track labels ----
    # Obstacle field: points + wheel position labels + outer ring samples
    obstacle_x = list(xs); obstacle_y = list(ys)
    for n in range(1, 13):
        a = np.deg2rad(90 - (n % 12) * 30)
        obstacle_x.append(R_LBL_A * np.cos(a)); obstacle_y.append(R_LBL_A * np.sin(a))
        obstacle_x.append(R_LBL_B * np.cos(a)); obstacle_y.append(R_LBL_B * np.sin(a))
    for theta in np.linspace(0, 2 * np.pi, 60, endpoint=False):
        obstacle_x.append(R_OUTER * np.cos(theta))
        obstacle_y.append(R_OUTER * np.sin(theta))

    R_LABEL_INIT = R_OUTER + 0.32
    # Spread initial label angles so co-angled points (e.g. inner+outer ring at same n)
    # don't start stacked on top of each other.
    base_angles = np.arctan2(ys, xs)
    order = np.argsort(base_angles)
    sorted_a = base_angles[order].astype(float).copy()
    MIN_GAP = np.deg2rad(7.0)
    for k in range(1, len(order)):
        if sorted_a[k] - sorted_a[k - 1] < MIN_GAP:
            sorted_a[k] = sorted_a[k - 1] + MIN_GAP
    label_angles = np.zeros_like(base_angles, dtype=float)
    for k, i in enumerate(order):
        label_angles[i] = sorted_a[k]

    texts = []
    for i in range(n_pts):
        artist = str(valid.iloc[i].get("Artist") or "").strip()
        title_ = str(valid.iloc[i].get("Track Title") or "").strip()
        label = label_format.format(artist=artist, title=title_)
        label = label.strip(" —-·").strip()
        if not label:
            label = f"Track {i + 1}"
        # Never truncate: the full track title is always shown.
        full = f"{i + 1}. {label}"

        lx = R_LABEL_INIT * np.cos(label_angles[i])
        ly = R_LABEL_INIT * np.sin(label_angles[i])

        t = ax.text(lx, ly, full,
                    fontsize=7.4, color=S["text"],
                    ha="center", va="center", zorder=5,
                    clip_on=False,   # never clip labels at the axes edge
                    path_effects=[patheffects.withStroke(
                        linewidth=S["text_stroke_w"], foreground=S["text_stroke"])])
        texts.append(t)

    adjust_text(
        texts, ax=ax,
        x=obstacle_x, y=obstacle_y,
        expand=(1.3, 1.5),
        arrowprops=dict(arrowstyle="-", color=S["leader"],
                        lw=0.5, alpha=0.85, shrinkA=2, shrinkB=4),
        target_x=list(xs), target_y=list(ys),
        force_text=(0.6, 0.8),
        force_static=(0.3, 0.4),
        force_pull=(0.0, 0.0),
        max_move=60,
        iter_lim=600,
    )

    # Post-process: project any label still inside the wheel back outside
    MIN_R = R_OUTER + 0.18
    for t in texts:
        px, py = t.get_position()
        r = np.hypot(px, py)
        if r < MIN_R:
            if r > 1e-6:
                t.set_position((px / r * MIN_R, py / r * MIN_R))
            else:
                t.set_position((0, MIN_R))


    # ---- colorbar (play order) ----
    sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cax = fig.add_axes([0.33, 0.055, 0.34, 0.013])
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.outline.set_linewidth(0.5); cbar.outline.set_edgecolor(S["cbar_outline"])
    cbar.ax.tick_params(labelsize=7.5, color=S["cbar_outline"],
                        labelcolor=S["cbar_text"], width=0.5)
    cbar.set_ticks([0, n_pts - 1])
    cbar.set_ticklabels(["1 (start)", f"{n_pts} (end)"])
    cbar.set_label("Play order", fontsize=8.5, color=S["cbar_text"], labelpad=4)

    # ---- save: PNG (raster) + PDF, SVG (vector, editable in Illustrator) ----
    # bbox_inches='tight' so long, untruncated labels are never cut off at the
    # figure edge; pad_inches keeps a small margin around them.
    output_path = Path(output_path)
    fig.savefig(output_path, dpi=300, facecolor=S["fig_bg"],
                bbox_inches="tight", pad_inches=0.15)        # PNG
    saved = {"png": output_path}
    for ext in (".pdf", ".svg"):
        p = output_path.with_suffix(ext)
        fig.savefig(p, facecolor=S["fig_bg"],
                    bbox_inches="tight", pad_inches=0.15)
        saved[ext.lstrip(".")] = p
    plt.show()
    return saved

## 6. 출력 폴더 & 예시 데이터

In [ ]:
def make_output_dir(base="./outputs", tag="camelot_journey"):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = Path(base) / f"{stamp}_{tag}"
    out.mkdir(parents=True, exist_ok=True)
    return out

In [ ]:



def example_setlist():
    rows = [
        ("Track 01", "Artist A", "Am"),
        ("Track 02", "Artist B", "Am"),
        ("Track 03", "Artist C", "Em"),
        ("Track 04", "Artist D", "G"),
        ("Track 05", "Artist E", "D"),
        ("Track 06", "Artist F", "Bm"),
        ("Track 07", "Artist G", "F#m"),
        ("Track 08", "Artist H", "F#m"),
        ("Track 09", "Artist I", "C#m"),
        ("Track 10", "Artist J", "E"),
        ("Track 11", "Artist K", "B"),
        ("Track 12", "Artist L", "G#m"),
        ("Track 13", "Artist M", "Ebm"),
        ("Track 14", "Artist N", "Ebm"),
        ("Track 15", "Artist O", "Bbm"),
        ("Track 16", "Artist P", "Fm"),
    ]
    df = pd.DataFrame(rows, columns=["Track Title", "Artist", "Key"])
    df["_camelot"] = df["Key"].apply(normalize_key)
    return df


df = example_setlist()
df.head()

## 7. 그리기

세 가지 디자인 중 하나를 골라 출력합니다.

| `style` | 느낌 |
|---|---|
| `"paper"` | 라이트 / 논문 차트 (viridis 그라데이션) |
| `"neon"` | 다크 클럽 무드 (네온 글로우, plasma) |
| `"mono"` | 미니멀 에디토리얼 (단색 웜 액센트, 가는 휠) |

실제 rekordbox txt를 쓸 때:

```python
df = parse_rekordbox_txt("./mysetlist.txt")
```

In [ ]:
# --- 세 디자인 미리보기: 아래 세 그림을 비교한 뒤 다음 셀에서 하나를 고르세요 ---
preview_dir = make_output_dir(base="./outputs", tag="previews")
for name in STYLES:
    print(f"=== design: {name} ===")
    plot_harmonic_journey(
        df,
        output_path=preview_dir / f"preview_{name}.png",
        subtitle=f"n = {len(df)} tracks",
        style=name,
    )

In [ ]:
# ▼▼▼ 여기서 디자인 하나를 고르세요 ▼▼▼
DESIGN = "paper"     # "paper" | "neon" | "mono"
# ▲▲▲ ───────────────────────── ▲▲▲

out_dir = make_output_dir(base="./outputs")
df.to_csv(out_dir / "tracklist.csv", index=False, encoding="utf-8-sig")

saved = plot_harmonic_journey(
    df,
    output_path=out_dir / f"camelot_journey_{DESIGN}.png",
    title="Harmonic Journey on the Camelot Wheel",
    subtitle=f"n = {len(df)} tracks",
    style=DESIGN,
)
print(f"\n선택한 디자인: {DESIGN}")
for fmt, path in saved.items():
    print(f"  {fmt.upper():4s}: {path}")

## 7-B. 튜닝 가이드

- **디자인 변경** → `DESIGN`을 `"paper"` / `"neon"` / `"mono"` 중 하나로
- **새 디자인 추가** → 위 `STYLES` 딕셔너리에 항목 하나 추가 (색·`cmap`·`glow` 등 지정)
- **컬러맵만 바꾸기** → 해당 스타일의 `cmap` 값을 `"cividis"`, `"magma"`, `"inferno"` 등으로
- **라벨이 안 보이거나 잘림** → 라벨은 절대 잘리지 않음(전체 제목 표시). 너무 빽빽하면 `LIM` 키우거나 `R_LABEL_INIT` 줄이기
- **라벨끼리 더 떨어뜨리고 싶음** → `MIN_GAP` 각도 키우기 (예: 10°), `expand`, `force_text` 키우기
- **leader 라인이 휠을 가로지름** → `max_move`을 줄여서 라벨이 점 근처에 머물게 하기

## 7-C. 일러스트레이터에서 편집

위에서 저장한 `.svg` 또는 `.pdf`를 일러스트레이터로 열면 됩니다.

- **SVG**: 가장 잘 보존됨. `svg.fonttype="none"` 덕분에 텍스트가 `<text>` 요소로 남아 폰트 교체, 글자 수정 모두 가능. 모든 도형이 개별 패스로 분리되어 있어 색·획·위치 자유 변경 가능
- **PDF**: `pdf.fonttype=42`로 TrueType 임베드되어 편집 가능. 일부 텍스트가 그룹화될 수 있음
- 시스템에 동일한 폰트(Menlo 등)가 없으면 일러스트레이터가 대체 폰트로 치환하는데, 이때 직접 원하는 폰트로 다시 지정하면 됩니다
- 곡선·화살표·점 모두 개별 객체로 살아 있어서 색상 통일, 화살표 스타일 변경, 라벨 위치 미세 조정 등이 모두 가능